In [ ]:
import sys
sys.path.append('..')
import metric_fit, parametrization, numpy as np, mesh
from tri_mesh_viewer import TriMeshViewer

m = mesh.Mesh('../examples/lilium.msh')
uv = np.loadtxt('data/lilium_tower_parametrization_4_10_2019_2.txt')

mflat = metric_fit.Mesh2D(uv, np.array(m.triangles()))
fitter = metric_fit.MetricFitter(mflat)
fitter.setTargetMetric([np.array([[1.5, 0.15], [0.15, 1]]) for i in range(mflat.numTris())])

In [ ]:
fitter.energy()

In [ ]:
fitter.bendingStiffness = 1e-3
fitter.setVars(fitter.getVars() + 1e-3 * np.random.uniform(low=-1, high=1, size=fitter.numVars()))

In [ ]:
import fd_validation
fd_validation.validateGrad(fitter, etype=metric_fit.MetricFitter.EnergyType.Full, fd_eps=1e-8)

In [ ]:
fd_validation.validateHessian(fitter)

In [ ]:
from tri_mesh_viewer import TriMeshViewer
immersedSurface = mesh.Mesh(fitter.getImmersion().transpose(), m.triangles())
viewer = TriMeshViewer(immersedSurface)
viewer.showWireframe()
viewer.show()

In [ ]:
import time
for i in range(10):
    metric_fit.fit_metric_newton(fitter, fitter.rigidMotionPinVars)
    immersedSurface = mesh.Mesh(fitter.getImmersion().transpose(), m.triangles())
    viewer.update(mesh=immersedSurface)
    time.sleep(0.01)